In [ ]:
import os
import sys
import json
import copy
import importlib
import numpy as np
import pandas as pd
import torch

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json"), "r") as f:
    config = json.load(f)

SEED = config["seed"]
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = config["paths"]["data_dir"]
FEATURES_DIR = config["paths"]["features_dir"]
RESULTS_DIR = config["paths"]["results_dir"]
SPLITS = ["train", "dev", "test"]
IN_DIMS = {"text": int(config.get("text_fusion_feature_dim", config["text_feature_dim"])), "audio": int(config.get("audio_fusion_feature_dim", config["audio_feature_dim"])), "video": config["video_feature_dim"]}
EMOTIONS = config["emotions"]
print(f"Device: {device} | input dims: {IN_DIMS}")

In [ ]:
MODULE_SRC = r'''
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from torch.utils.data import Dataset, DataLoader

PAD_SPEAKER = 0     
UNK_SPEAKER = 1     
FEATURE_FILES = {"text": "text_features_{split}.npy", "audio": "audio_features_{split}.npy", "video": "video_features_{split}.npy"}
def _resolve_text_file():
    try:
        import json as _json
        with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "config.json")) as _fh:
            return _json.load(_fh).get("text_feature_file") or "text_features_{split}.npy"
    except (OSError, ValueError):
        return "text_features_{split}.npy"
FEATURE_FILES["text"] = _resolve_text_file()
def _resolve_audio_file():
    try:
        import json as _json
        with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "config.json")) as _fh:
            return _json.load(_fh).get("audio_feature_file") or "audio_features_{split}.npy"
    except (OSError, ValueError):
        return "audio_features_{split}.npy"
FEATURE_FILES["audio"] = _resolve_audio_file()
VALID_FILES = {"audio": "audio_valid_{split}.npy", "video": "video_valid_{split}.npy"}


def fit_speaker_vocab(train_df, min_count=50):
    counts = train_df["Speaker"].value_counts()
    names = [n for n, c in counts.items() if c >= min_count]
    return {name: i + 2 for i, name in enumerate(names)}


def build_context_manifest(df, global_vocab=None, max_local_speakers=12):
    if not isinstance(df.index, pd.RangeIndex):
        raise ValueError("DataFrame must have a default RangeIndex (feature arrays are aligned to CSV row order).")
    runs = int((df["Dialogue_ID"] != df["Dialogue_ID"].shift()).sum())
    if runs != df["Dialogue_ID"].nunique():
        raise ValueError("Dialogues are not contiguous blocks in CSV row order.")

    grp = df.groupby("Dialogue_ID", sort=False)
    pos = grp.cumcount().to_numpy()
    dlg_len = grp["Dialogue_ID"].transform("size").to_numpy()

    local = np.zeros(len(df), dtype=np.int64)
    for _, g in grp:
        order = {}
        ids = []
        for sp in g["Speaker"]:
            if sp not in order:
                order[sp] = len(order) + 1
            ids.append(order[sp])
        if len(order) > max_local_speakers:
            raise ValueError(f"Dialogue with {len(order)} distinct speakers exceeds max_local_speakers={max_local_speakers}")
        local[g.index.to_numpy()] = ids

    manifest = pd.DataFrame({
        "row_idx": np.arange(len(df)),
        "Dialogue_ID": df["Dialogue_ID"].to_numpy(),
        "Utterance_ID": df["Utterance_ID"].to_numpy(),
        "pos": pos,
        "dlg_len": dlg_len,
        "speaker": df["Speaker"].to_numpy(),
        "speaker_local": local,
    })
    if global_vocab is not None:
        manifest["speaker_global"] = [global_vocab.get(s, UNK_SPEAKER) for s in df["Speaker"]]
    return manifest


def load_split_features(features_dir, split, expected_dims):
    feats, valid = {}, {}
    for m, dim in expected_dims.items():
        path = os.path.join(features_dir, FEATURE_FILES[m].format(split=split))
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing {path} -- run the feature-extraction phase for '{m}' first.")
        x = np.load(path).astype(np.float32, copy=False)
        if x.ndim != 2 or x.shape[1] != dim:
            raise ValueError(f"{path}: shape {x.shape}, expected (N, {dim})")
        if not np.isfinite(x).all():
            raise ValueError(f"{path}: contains non-finite values")
        feats[m] = x
        if m in VALID_FILES:
            vpath = os.path.join(features_dir, VALID_FILES[m].format(split=split))
            if not os.path.exists(vpath):
                raise FileNotFoundError(f"Missing validity mask {vpath}")
            v = np.load(vpath).astype(bool)
            if v.shape != (x.shape[0],):
                raise ValueError(f"{vpath}: shape {v.shape} does not match features {x.shape}")
            valid[m] = v
        else:
            valid[m] = np.ones(x.shape[0], dtype=bool)
    return feats, valid


class MELDDialogueDataset(Dataset):
    def __init__(self, df, manifest, features, valid, emotions, speaker_mode="local"):
        n = len(df)
        if len(manifest) != n or not (manifest["row_idx"].to_numpy() == np.arange(n)).all():
            raise ValueError("Manifest is not aligned with the CSV rows.")
        for m, x in features.items():
            if x.shape[0] != n or valid[m].shape[0] != n:
                raise ValueError(f"Feature array for '{m}' has {x.shape[0]} rows but the CSV has {n}.")
        label_map = {e: i for i, e in enumerate(emotions)}
        labels = df["Emotion"].map(label_map)
        if labels.isna().any():
            raise KeyError(f"Unknown emotion labels: {sorted(set(df['Emotion'][labels.isna()]))}")
        col = {"local": "speaker_local", "global": "speaker_global"}[speaker_mode]
        if col not in manifest.columns:
            raise KeyError(f"Manifest has no '{col}' column (speaker_mode='{speaker_mode}').")
        self.features, self.valid = features, valid
        self.labels = labels.to_numpy(np.int64).copy()
        self.speaker_ids = manifest[col].to_numpy(np.int64).copy()
        self.dialogue_ids = manifest["Dialogue_ID"].to_numpy()
        starts = np.flatnonzero(manifest["pos"].to_numpy() == 0)
        lens = manifest["dlg_len"].to_numpy()[starts]
        self.blocks = [(int(s), int(l)) for s, l in zip(starts, lens)]
        assert sum(l for _, l in self.blocks) == n

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, i):
        s, n = self.blocks[i]
        sl = slice(s, s + n)
        return {
            "dialogue_id": int(self.dialogue_ids[s]),
            "row_idx": torch.arange(s, s + n, dtype=torch.long),
            "labels": torch.from_numpy(self.labels[sl]),
            "speaker_ids": torch.from_numpy(self.speaker_ids[sl]),
            "features": {m: torch.from_numpy(x[sl]) for m, x in self.features.items()},
            "valid": {m: torch.from_numpy(v[sl]) for m, v in self.valid.items()},
        }


def collate_dialogues(items):
    B = len(items)
    L = max(len(it["row_idx"]) for it in items)
    mods = list(items[0]["features"].keys())
    out = {
        "dialogue_id": torch.tensor([it["dialogue_id"] for it in items]),
        "lengths": torch.tensor([len(it["row_idx"]) for it in items]),
        "row_idx": torch.full((B, L), -1, dtype=torch.long),
        "labels": torch.full((B, L), -100, dtype=torch.long),
        "speaker_ids": torch.full((B, L), PAD_SPEAKER, dtype=torch.long),
        "mask": torch.zeros(B, L, dtype=torch.bool),
        "features": {m: torch.zeros(B, L, items[0]["features"][m].shape[1]) for m in mods},
        "valid": {m: torch.ones(B, L, dtype=torch.bool) for m in mods},
    }
    for b, it in enumerate(items):
        n = len(it["row_idx"])
        out["row_idx"][b, :n] = it["row_idx"]
        out["labels"][b, :n] = it["labels"]
        out["speaker_ids"][b, :n] = it["speaker_ids"]
        out["mask"][b, :n] = True
        for m in mods:
            out["features"][m][b, :n] = it["features"][m]
            out["valid"][m][b, :n] = it["valid"][m]
    return out


def make_loader(dataset, batch_size=16, shuffle=False, seed=42, num_workers=0):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_dialogues,
                      generator=g, num_workers=num_workers)


def batch_to_device(batch, device):
    return {
        k: ({m: t.to(device) for m, t in v.items()} if isinstance(v, dict) else v.to(device))
        for k, v in batch.items()
    }


class SequentialPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)[:, : d_model // 2]
        self.register_buffer("pe", pe, persistent=False)
        self.max_len = max_len

    def forward(self, length):
        if length > self.max_len:
            raise ValueError(f"Dialogue length {length} exceeds max_len={self.max_len}")
        return self.pe[:length].unsqueeze(0)


class ModalityContextEncoder(nn.Module):
    def __init__(self, in_dim, d_model=256, n_heads=4, n_layers=2, ff_mult=4, dropout=0.3,
                 num_speaker_ids=13, max_len=64):
        super().__init__()
        self.in_dim, self.d_model = in_dim, d_model
        self.proj = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model))
        self.missing_emb = nn.Parameter(torch.randn(d_model) * 0.02)  
        self.speaker_emb = nn.Embedding(num_speaker_ids, d_model, padding_idx=PAD_SPEAKER)
        self.pos = SequentialPositionalEncoding(d_model, max_len)
        self.in_dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * ff_mult, dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, norm=nn.LayerNorm(d_model),
                                                 enable_nested_tensor=False)

    def forward(self, x, speaker_ids, mask, valid=None):
        h = self.proj(x)
        if valid is not None:
            h = torch.where(valid.unsqueeze(-1), h, self.missing_emb.expand_as(h))
        h = h + self.speaker_emb(speaker_ids) + self.pos(h.size(1))
        h = self.in_dropout(h)
        out = self.transformer(h, src_key_padding_mask=~mask)
        return out * mask.unsqueeze(-1)


class MultimodalContextEncoder(nn.Module):
    def __init__(self, in_dims, modalities=None, **encoder_kwargs):
        super().__init__()
        self.modalities = list(modalities) if modalities is not None else list(in_dims.keys())
        self.encoders = nn.ModuleDict({m: ModalityContextEncoder(in_dims[m], **encoder_kwargs) for m in self.modalities})

    def forward(self, batch):
        return {m: self.encoders[m](batch["features"][m], batch["speaker_ids"], batch["mask"], batch["valid"][m])
                for m in self.modalities}


@torch.no_grad()
def collect_aligned(encoder, loader, n_rows, device):
    was_training = encoder.training
    encoder.eval()
    bufs, seen = {}, np.zeros(n_rows, dtype=np.int64)
    for batch in loader:
        batch = batch_to_device(batch, device)
        outs = encoder(batch)
        sel = batch["mask"]
        rows = batch["row_idx"][sel].cpu().numpy()
        for m, o in outs.items():
            if m not in bufs:
                bufs[m] = np.full((n_rows, o.size(-1)), np.nan, dtype=np.float32)
            bufs[m][rows] = o[sel].cpu().numpy()
        np.add.at(seen, rows, 1)
    if not (seen == 1).all():
        raise RuntimeError(f"Row coverage error: {int((seen != 1).sum())} rows produced != 1 time")
    if any(np.isnan(b).any() for b in bufs.values()):
        raise RuntimeError("NaNs in aligned context export")
    encoder.train(was_training)
    return bufs
'''

module_path = os.path.join(BASE_DIR, "meld_context.py")
with open(module_path, "w") as f:
    f.write(MODULE_SRC)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
import meld_context
importlib.reload(meld_context)
from meld_context import (build_context_manifest, fit_speaker_vocab, load_split_features, MELDDialogueDataset,
                          make_loader, batch_to_device, collect_aligned, MultimodalContextEncoder,
                          ModalityContextEncoder, PAD_SPEAKER, UNK_SPEAKER)
print(f"Wrote and imported {module_path}")

In [ ]:
dfs = {s: pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv")) for s in SPLITS}
max_len_seen = max(int(d.groupby("Dialogue_ID", sort=False).size().max()) for d in dfs.values())
max_spk_seen = max(int(d.groupby("Dialogue_ID")["Speaker"].nunique().max()) for d in dfs.values())
print(f"Longest dialogue: {max_len_seen} utterances | most speakers in one dialogue: {max_spk_seen}")
print(f"Distinct speaker names (train): {dfs['train']['Speaker'].nunique()}")

MAX_LEN = 64
MAX_LOCAL_SPEAKERS = 12
assert max_len_seen <= MAX_LEN and max_spk_seen <= MAX_LOCAL_SPEAKERS

speaker_vocab = fit_speaker_vocab(dfs["train"], min_count=50)
print(f"Global speaker vocabulary (train, >=50 utterances): {len(speaker_vocab)} speakers (+ pad, unk)")

In [ ]:
manifests = {}
for s in SPLITS:
    m = build_context_manifest(dfs[s], global_vocab=speaker_vocab, max_local_speakers=MAX_LOCAL_SPEAKERS)
    manifests[s] = m
    m.to_csv(os.path.join(DATA_DIR, f"context_manifest_{s}.csv"), index=False)

    assert (m["row_idx"].values == np.arange(len(dfs[s]))).all()
    assert (m["Dialogue_ID"].values == dfs[s]["Dialogue_ID"].values).all()
    assert (m["Utterance_ID"].values == dfs[s]["Utterance_ID"].values).all()
    for did, g in m.groupby("Dialogue_ID", sort=False):
        assert g["pos"].tolist() == list(range(len(g))), f"[{s}] dialogue {did}: pos is not 0..n-1 in row order"
        assert g["dlg_len"].eq(len(g)).all()
    n_gap = int(sum((g["pos"].values != g["Utterance_ID"].values).any() for _, g in m.groupby("Dialogue_ID", sort=False)))
    print(f"[{s}] manifest saved: {len(m)} rows | dialogues where position != Utterance_ID (gaps handled): {n_gap}")

with open(os.path.join(DATA_DIR, "speaker_vocab.json"), "w") as f:
    json.dump(speaker_vocab, f, indent=2)

In [ ]:
features, valid = {}, {}
for s in SPLITS:
    features[s], valid[s] = load_split_features(FEATURES_DIR, s, IN_DIMS)
    n = len(dfs[s])
    assert all(x.shape == (n, IN_DIMS[m]) for m, x in features[s].items())
    unusable = {m: int((~v).sum()) for m, v in valid[s].items()}
    print(f"[{s}] features loaded for {n} rows; unusable clips per modality: {unusable}")

In [ ]:
CONTEXT_CFG = {
    "d_model": 256, "n_heads": 4, "n_layers": 2, "ff_mult": 4, "dropout": 0.3,
    "speaker_mode": "local",          
    "max_len": MAX_LEN, "max_local_speakers": MAX_LOCAL_SPEAKERS,
}
def num_speaker_ids(mode):
    return (MAX_LOCAL_SPEAKERS + 1) if mode == "local" else (len(speaker_vocab) + 2)

def build_loader(split, mode="local", batch_size=16, shuffle=False, feats=None):
    ds = MELDDialogueDataset(dfs[split], manifests[split], feats or features[split], valid[split], EMOTIONS, speaker_mode=mode)
    return ds, make_loader(ds, batch_size=batch_size, shuffle=shuffle, seed=SEED)

def build_model(mode="local", modalities=None):
    return MultimodalContextEncoder(
        IN_DIMS, modalities=modalities, d_model=CONTEXT_CFG["d_model"], n_heads=CONTEXT_CFG["n_heads"],
        n_layers=CONTEXT_CFG["n_layers"], ff_mult=CONTEXT_CFG["ff_mult"], dropout=CONTEXT_CFG["dropout"],
        num_speaker_ids=num_speaker_ids(mode), max_len=MAX_LEN)

train_ds, train_loader = build_loader("train", shuffle=True)
batch = next(iter(train_loader))
print({k: (tuple(v.shape) if torch.is_tensor(v) else {m: tuple(t.shape) for m, t in v.items()}) for k, v in batch.items()})

torch.manual_seed(SEED)
model = build_model().to(device)
for m, enc in model.encoders.items():
    print(f"{m:5s} encoder: {sum(p.numel() for p in enc.parameters()):,} parameters")
print(f"Total: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
results = []
def check(name, cond, detail=""):
    results.append({"test": name, "passed": bool(cond), "detail": str(detail)})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

def run(model_, b, train=False):
    model_.train(train)
    with torch.set_grad_enabled(train):
        return model_(batch_to_device(b, device))

eval_ds, eval_loader = build_loader("train", batch_size=16)
torch.manual_seed(SEED)
model = build_model().to(device).eval()
b = next(iter(eval_loader))
out = run(model, b)

B, L = b["mask"].shape
ok = all(o.shape == (B, L, CONTEXT_CFG["d_model"]) for o in out.values())
pad_zero = all((o[~b["mask"].to(device)] == 0).all().item() for o in out.values())
check("T1 output shapes [B,L,d_model] and zero padding", ok and pad_zero, f"B={B}, L={L}")

lens = [len(it) for it in [eval_ds[i]["row_idx"] for i in range(len(eval_ds))]]
i_short = int(np.argmin(np.where(np.array(lens) >= 2, lens, 10**6)))
i_long = int(np.argmax(lens))
from meld_context import collate_dialogues
alone = collate_dialogues([eval_ds[i_short]])
padded = collate_dialogues([eval_ds[i_short], eval_ds[i_long]])
oa, op = run(model, alone), run(model, padded)
n_s = lens[i_short]
diff = max((oa[m][0, :n_s] - op[m][0, :n_s]).abs().max().item() for m in model.modalities)
check("T2 padding invariance", diff < 1e-4, f"max |diff|={diff:.2e}, short={n_s}, long={lens[i_long]}")

i_p = next(i for i in range(len(eval_ds)) if 4 <= lens[i] <= 8)
single = collate_dialogues([eval_ds[i_p]])
n_p = lens[i_p]
rev = copy.deepcopy(single)
for key in ("speaker_ids", "mask"):
    rev[key] = rev[key].flip(1)
for key in ("features", "valid"):
    rev[key] = {m: t.flip(1) for m, t in rev[key].items()}
o1, o2 = run(model, single), run(model, rev)
d_with = max((o1[m].flip(1) - o2[m]).abs().max().item() for m in model.modalities)
model_nopos = copy.deepcopy(model)
for enc in model_nopos.encoders.values():
    enc.pos.pe.zero_()
p1, p2 = run(model_nopos, single), run(model_nopos, rev)
d_without = max((p1[m].flip(1) - p2[m]).abs().max().item() for m in model.modalities)
check("T3 positional encoding makes outputs order-sensitive", d_with > 1e-3 and d_without < 1e-4,
      f"with PE diff={d_with:.3f}; control (PE zeroed) diff={d_without:.1e}")

spk_changed = copy.deepcopy(single)
spk_changed["speaker_ids"] = torch.where(spk_changed["mask"], torch.ones_like(spk_changed["speaker_ids"]), spk_changed["speaker_ids"])
d_spk = max((run(model, single)[m] - run(model, spk_changed)[m]).abs().max().item() for m in model.modalities)
check("T4 speaker identity changes the representation", d_spk > 1e-3, f"max |diff|={d_spk:.3f}")

mt = manifests["train"]
gap_dialogues = [did for did, g in mt.groupby("Dialogue_ID", sort=False) if (g["pos"].values != g["Utterance_ID"].values).any()]
g0 = mt[mt["Dialogue_ID"] == gap_dialogues[0]]
check("T5 gap dialogues: position = row order (not Utterance_ID)",
      len(gap_dialogues) > 0 and g0["pos"].tolist() == list(range(len(g0))) and g0["Utterance_ID"].tolist() != list(range(len(g0))),
      f"{len(gap_dialogues)} gap dialogues, e.g. Dialogue_ID={gap_dialogues[0]}: Utterance_ID={g0['Utterance_ID'].tolist()}")
check("T5b model inputs expose no Utterance_ID", "Utterance_ID" not in b and "utterance_id" not in b)

n_rows = len(dfs["train"])
aligned = collect_aligned(model, eval_loader, n_rows, device)
check("T6a aligned export covers every CSV row exactly once", all(a.shape == (n_rows, CONTEXT_CFG["d_model"]) for a in aligned.values()))
rng = np.random.RandomState(0)
uniq = dfs["train"]["Dialogue_ID"].unique()
sample = rng.choice(uniq, size=min(60, len(uniq)), replace=False)
worst = 0.0
for did in sample:
    rows = np.flatnonzero(dfs["train"]["Dialogue_ID"].values == did)        
    single_b = {
        "features": {m: torch.from_numpy(features["train"][m][rows])[None] for m in IN_DIMS},
        "valid": {m: torch.from_numpy(valid["train"][m][rows])[None] for m in IN_DIMS},
        "speaker_ids": torch.from_numpy(mt["speaker_local"].values[rows])[None],
        "mask": torch.ones(1, len(rows), dtype=torch.bool),
    }
    direct = run(model, single_b)
    worst = max(worst, max(np.abs(direct[m][0].cpu().numpy() - aligned[m][rows]).max() for m in model.modalities))
check(f"T6b aligned rows equal direct per-dialogue computation ({len(sample)} sampled dialogues)", worst < 1e-4, f"max |diff|={worst:.2e}")

r = int(rows[len(rows) // 2])
d_r = dfs["train"].at[r, "Dialogue_ID"]
perturbed = {m: x.copy() for m, x in features["train"].items()}
perturbed["text"][r] += 5.0
_, loader_p = build_loader("train", batch_size=16, feats=perturbed)
aligned_p = collect_aligned(model, loader_p, n_rows, device)
same_dlg = (dfs["train"]["Dialogue_ID"].values == d_r)
outside = np.abs(aligned_p["text"][~same_dlg] - aligned["text"][~same_dlg]).max()
inside = np.abs(aligned_p["text"][same_dlg] - aligned["text"][same_dlg]).max()
other_mod = np.abs(aligned_p["audio"] - aligned["audio"]).max()
check("T6c row-level alignment: perturbation stays inside its own dialogue and modality",
      outside < 1e-6 and inside > 1e-3 and other_mod < 1e-6, f"outside={outside:.1e}, inside={inside:.3f}, other modality={other_mod:.1e}")

forced = copy.deepcopy(single)
forced["valid"] = {m: t.clone() for m, t in forced["valid"].items()}
forced["valid"]["audio"][0, 1] = False
base = run(model, forced)
tampered = copy.deepcopy(forced)
tampered["features"] = {m: t.clone() for m, t in tampered["features"].items()}
tampered["features"]["audio"][0, 1] += 100.0
d_miss = (run(model, tampered)["audio"] - base["audio"]).abs().max().item()
check("T8 unusable-clip features are ignored (replaced by the learned missing token)", d_miss < 1e-6, f"max |diff|={d_miss:.1e}")

forced_all = copy.deepcopy(single)      
forced_all["valid"] = {m: t.clone() for m, t in forced_all["valid"].items()}
for m_ in forced_all["valid"]:
    forced_all["valid"][m_][0, 1] = False
model.train()
loss = sum(o.pow(2).mean() for o in run(model, forced_all, train=True).values())
loss.backward()
no_grad = [n for n, p in model.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
dead = [n for n, p in model.named_parameters() if p.grad is not None and p.grad.abs().sum() == 0 and "speaker_emb" not in n]
check("T7 gradients reach every parameter (incl. speaker + missing-token embeddings)", not no_grad and not dead, f"no_grad={no_grad[:3]}, zero_grad={dead[:3]}")
model.zero_grad(); model.eval()

torch.manual_seed(SEED); m1 = build_model()
torch.manual_seed(SEED); m2 = build_model()
same_init = all(torch.equal(a, b_) for a, b_ in zip(m1.state_dict().values(), m2.state_dict().values()))
m1.to(device).eval()
e1, e2 = run(m1, single), run(m1, single)
det_eval = all(torch.equal(e1[m], e2[m]) for m in m1.modalities)
t1, t2 = run(m1, single, train=True), run(m1, single, train=True)
stoch_train = any((t1[m] - t2[m]).abs().max() > 1e-6 for m in m1.modalities)
check("T9 same seed -> same init; eval deterministic; train-mode dropout is stochastic", same_init and det_eval and stoch_train)

_wavlm = str(config.get("audio_feature_set", "")).startswith("wavlm")
_faces = str(config.get("video_backbone", "")).startswith("mtcnn")
_expected = {"text": int(config.get("text_fusion_feature_dim", 768)), "audio": int(config.get("audio_fusion_feature_dim", 768)) if _wavlm else 88, "video": 1280 if _faces else 1024}
_on_disk = {m: int(np.load(os.path.join(FEATURES_DIR, meld_context.FEATURE_FILES[m].format(split="train")), mmap_mode="r").shape[1]) for m in IN_DIMS}
check("T10a input dims match the declared extractors and the cached feature files", IN_DIMS == _expected and _on_disk == _expected,
      f"config={IN_DIMS}, expected={_expected}, on disk={_on_disk}")
_usable = {s: {m: float(v.mean()) for m, v in valid[s].items()} for s in SPLITS}
check("T10d every modality has usable clips in every split (shares printed in the detail)", all(x > 0 for d in _usable.values() for x in d.values()),
      {s: {m: round(x, 3) for m, x in d.items()} for s, d in _usable.items()})
uni = build_model(modalities=["audio"]).to(device).eval()
ou = run(uni, b)
check("T10b unimodal encoder (audio only) runs on a full batch", list(ou.keys()) == ["audio"] and ou["audio"].shape[-1] == CONTEXT_CFG["d_model"])
_, loader_g = build_loader("train", mode="global", batch_size=16)
bg = next(iter(loader_g))
glob = build_model(mode="global").to(device).eval()
og = run(glob, bg)
check("T10c global speaker-vocabulary mode runs", all(torch.isfinite(o).all() for o in og.values()) and int(bg["speaker_ids"].max()) < len(speaker_vocab) + 2,
      f"max id={int(bg['speaker_ids'].max())}, ids available={len(speaker_vocab) + 2}")

failed = [r_ for r_ in results if not r_["passed"]]
with open(os.path.join(RESULTS_DIR, "context_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r_['test'] for r_ in failed]}"
print(f"\nAll {len(results)} verification checks passed.")

In [ ]:
context_config = dict(CONTEXT_CFG)
context_config.update({
    "in_dims": IN_DIMS,
    "num_speaker_ids_local": num_speaker_ids("local"),
    "num_speaker_ids_global": num_speaker_ids("global"),
    "module": "meld_context.py",
    "note": "Position = order within the dialogue in CSV row order; bidirectional attention within a dialogue.",
})
with open(os.path.join(BASE_DIR, "context_config.json"), "w") as f:
    json.dump(context_config, f, indent=4)

for s in SPLITS:
    assert os.path.exists(os.path.join(DATA_DIR, f"context_manifest_{s}.csv"))
assert os.path.exists(module_path) and os.path.exists(os.path.join(BASE_DIR, "context_config.json"))
print("Phase 6 Context Modeling completed successfully.")